In [3]:
import csv
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ================= CONFIG =================
OUTPUT_FILE = "kppp_cmnl_all_pages.csv"
DELAY = 2  # seconds delay after you press ENTER before scraping
HEADERS = [
    "Sl No", "Department", "Office", "Tender ID", "Work Name",
    "Tender Type", "Work Category", "Tender Value", "Publish Date",
    "Closing Date", "Document Type", "Remarks"
]
# ===========================================

# Start browser
driver = webdriver.Chrome()
driver.maximize_window()
driver.get("https://kppp.karnataka.gov.in/#/portal/searchTender/live")

wait = WebDriverWait(driver, 20)

print("\n>>> Solve CAPTCHA, apply your filters manually, click 'Works' tab, then press ENTER...")
input()  # wait for you to be ready

all_data = []

while True:
    try:
        # --- Ensure Works tab is still active ---
        try:
            works_tab = driver.find_element(By.ID, "stepperTabLabel1")
            if "active" not in works_tab.get_attribute("class"):
                works_tab.click()
                time.sleep(1)
                print("✅ Works tab re-activated")
            else:
                print("✅ Works tab active")
        except:
            print("[WARN] Could not check Works tab.")

        # --- Wait a bit to allow table to load after manual NEXT ---
        time.sleep(DELAY)

        # --- Scrape table rows ---
        try:
            rows = wait.until(EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "table tbody tr")
            ))
            page_data = []
            for idx, row in enumerate(rows, 1):
                cells = [c.text.strip() for c in row.find_elements(By.TAG_NAME, "td")]
                if not any(cells):  # skip blank rows
                    continue
                print(f"  Row {idx}: {cells}")
                all_data.append(cells)
                page_data.append(cells)

            if not page_data:
                print("[INFO] No data found on this page.")

        except Exception as e:
            print(f"[WARN] Could not scrape this page: {e}")

    except Exception as e:
        print(f"[ERROR] {e}")

    # --- Ask user if they want to go to next page ---
    user_input = input("\nClick NEXT manually in browser, then press ENTER for next page (or type 'q' to quit): ").strip().lower()
    if user_input == "q":
        break

# --- Save to CSV ---
try:
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(HEADERS)  # write headers
        writer.writerows(all_data)
    print(f"\n✅ Data saved to {OUTPUT_FILE}")
except PermissionError:
    print(f"[ERROR] Permission denied while saving to {OUTPUT_FILE}. Close the file and try again.")

driver.quit()



>>> Solve CAPTCHA, apply your filters manually, click 'Works' tab, then press ENTER...


✅ Works tab active
  Row 21: ['1', 'Infrastructure Development Ports And Inland Water Transport Department', 'Executive engineer Udupi', 'IDD/2024-25/OW/WORK_INDENT136', 'Maintenance of North Side Breakwater at Koderi, Byndoor Taluk, Udupi District.', 'Open', 'Other Works', '---', '23-06-2025 16:25:18', '01-07-2025 17:00:00', 'Request for Quotation (RFQ)', '']
  Row 22: ['2', 'Infrastructure Development Ports And Inland Water Transport Department', 'Executive engineer Udupi', 'IDD/2024-25/FP/WORK_INDENT112/CALL-3', 'CONSTRUCTION OF EMERGENT SEA EROSION PROTECTION WORK AT UCHILA IN SOMESHWAR TOWN MUNCIPAL OF DAKSHINA KANNADA DISTRICT (CH. 2.460 KM to 2.545 KM)', 'Open', 'Flood Protection Works', '---', '17-06-2025 11:38:29', '25-06-2025 17:00:00', 'Request for Quotation (RFQ)', '']
  Row 23: ['3', 'Infrastructure Development Ports And Inland Water Transport Department', 'Executive engineer Udupi', 'IDD/2024-25/FP/WORK_INDENT110/CALL-3', 'CONSTRUCTION OF EMERGENT SEA EROSION PROTECTION W


Click NEXT manually in browser, then press ENTER for next page (or type 'q' to quit):  


✅ Works tab active
  Row 21: ['21', 'Infrastructure Development Ports And Inland Water Transport Department', 'Executive engineer Udupi', 'IDD/2024-25/FP/WORK_INDENT97', 'Rip-rap Emergency Preventive Work for sea erosion on the beach at Alivekodi Ram Bhajana Mandira in Byndur Taluk, Udupi District.', 'Open', 'Flood Protection Works', '---', '19-03-2025 17:51:38', '27-03-2025 17:00:00', 'Request for Quotation (RFQ)', '']
  Row 22: ['22', 'Infrastructure Development Ports And Inland Water Transport Department', 'Executive engineer Udupi', 'IDD/2024-25/FP/WORK_INDENT96', 'Rip-rap emergency prevention work for the sea beach on the sea shore near the house of Adragoli Dotti Pujarti of Kirimanjeshwar village, Byndur taluk, Udupi district.', 'Open', 'Flood Protection Works', '---', '19-03-2025 17:50:58', '27-03-2025 17:00:00', 'Request for Quotation (RFQ)', '']
  Row 23: ['23', 'Infrastructure Development Ports And Inland Water Transport Department', 'Executive engineer Udupi', 'IDD/2024-25/F


Click NEXT manually in browser, then press ENTER for next page (or type 'q' to quit):  q



✅ Data saved to kppp_cmnl_all_pages.csv


In [3]:
import csv
import time
import pandas as pd
from pathlib import Path

from selenium import webdriver
from selenium.webdriver import ChromeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException


# ---- CONFIG ----
START_URL = "https://kppp.karnataka.gov.in/#/portal/searchTender/live"
INPUT_CSV = "KPPP_ALL_DATA_PAST.csv"
OUT_CSV = Path("KPPP_ALL_DATA_BIDDER_PAST.csv")


def make_driver(headless=False, detach=True):
    opts = ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    if detach:
        opts.add_experimental_option("detach", True)
    d = webdriver.Chrome(options=opts)
    d.set_page_load_timeout(120)
    return d


def safe_click(drv, element):
    """Click with retry using JS if intercepted"""
    drv.execute_script("arguments[0].scrollIntoView({block:'center'});", element)
    try:
        element.click()
    except Exception:
        drv.execute_script("arguments[0].click();", element)


def wait_visible(drv, locator, timeout=25):
    return WebDriverWait(drv, timeout).until(EC.visibility_of_element_located(locator))


def safe_text(el):
    try:
        return (el.text or "").strip()
    except Exception:
        return ""


# ---------------- Search ----------------
def search_tender_by_id(drv, tender_id):
    inp = wait_visible(drv, (By.ID, "tendernumberInputField"), timeout=15)
    inp.clear()
    inp.send_keys(tender_id)

    # Category = Works
    cat_sel = Select(drv.find_element(By.ID, "categorySelect"))
    cat_sel.select_by_visible_text("Works")

    # Status = Awarded
    st_sel = Select(drv.find_element(By.ID, "tenderstatusdropdownSelect"))
    for opt in st_sel.options:
        if "awarded" in (opt.text or "").lower():
            st_sel.select_by_visible_text(opt.text)
            break

    # Search
    safe_click(drv, drv.find_element(By.XPATH, "//button[normalize-space()='Search']"))

    try:
        wait_visible(drv, (By.CSS_SELECTOR, "table#works tbody tr"), timeout=20)
        return True
    except TimeoutException:
        return False


# ---------------- Scraping ----------------
def open_view_selected_bidders(drv):
    """Always re-find row & kebab, retry if stale"""
    for attempt in range(3):
        try:
            row = WebDriverWait(drv, 20).until(
                EC.presence_of_element_located((By.XPATH, "(//table[@id='works']//tbody/tr)[1]"))
            )
            kebab_toggle = row.find_element(By.XPATH, ".//img[contains(@src,'kebab_Icon.svg')]/ancestor::a[1]")
            safe_click(drv, kebab_toggle)

            view_bidders = WebDriverWait(drv, 10).until(
                EC.presence_of_element_located((By.XPATH,
                    "//ul[contains(@class,'dropdown')]//img[contains(@src,'view_selected_suppliers_for_this_tender_Icon.svg')]/ancestor::lib-icon"
                ))
            )
            safe_click(drv, view_bidders)
            return
        except StaleElementReferenceException:
            print("⚠️ Stale row, retrying...")
            time.sleep(1)
    raise Exception("Could not click 'View Selected Bidders'")


def scrape_bidders_from_popup(drv, tender_id):
    wait_visible(drv, (By.ID, "detailsofSelectedBidCard"), timeout=25)

    rows = WebDriverWait(drv, 15).until(
        EC.presence_of_all_elements_located(
            (By.XPATH, "//div[@id='detailsofSelectedBidCard']//table//tbody/tr")
        )
    )

    bidder_names, company_names, bid_numbers = [], [], []
    for r in rows:
        cells = r.find_elements(By.TAG_NAME, "td")
        if len(cells) >= 4:
            bname = safe_text(cells[1])
            cname = safe_text(cells[2])
            bnum = safe_text(cells[3])
            if bname or cname or bnum:
                bidder_names.append(bname)
                company_names.append(cname)
                bid_numbers.append(bnum)

    # Close popup
    try:
        close_btn = WebDriverWait(drv, 10).until(EC.element_to_be_clickable((By.ID, "closebuttonButton")))
        safe_click(drv, close_btn)
        WebDriverWait(drv, 10).until(EC.invisibility_of_element_located((By.ID, "detailsofSelectedBidCard")))
    except Exception:
        pass

    time.sleep(1)  # small delay for DOM reset
    return bidder_names, company_names, bid_numbers


# ---------------- Main ----------------
def main():
    drv = make_driver(headless=False, detach=True)
    try:
        drv.get(START_URL)
        input(">>> Solve CAPTCHA if it appears, then press ENTER here to continue... ")

        ids_df = pd.read_csv(INPUT_CSV)

        OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
        with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "Department","Office","Work Name","Tender Type","Tender ID",
                "Bidder Name(s)","Company Name(s)","Bid Number(s)","Status"
            ])

            for _, row in ids_df.iterrows():
                t_id = str(row["Tender ID"]).strip()
                dept = row.get("Department", "")
                office = row.get("Office", "")
                work_name = row.get("Work Name", "")
                tender_type = row.get("Tender Type", "")

                print(f"\n=== Processing {t_id} ===")
                try:
                    if not search_tender_by_id(drv, t_id):
                        writer.writerow([dept, office, work_name, tender_type, t_id, "", "", "", "Not Found"])
                        print(f"[INFO] Tender {t_id} not found.")
                        continue

                    open_view_selected_bidders(drv)
                    bidder_names, company_names, bid_numbers = scrape_bidders_from_popup(drv, t_id)

                    status = "Success" if bidder_names else "No bidders scraped"
                    writer.writerow([
                        dept, office, work_name, tender_type, t_id,
                        " | ".join(bidder_names),
                        " | ".join(company_names),
                        " | ".join(bid_numbers),
                        status
                    ])
                    print(f"[OK] {t_id} → {status} ({len(bidder_names)} rows).")

                except Exception as e:
                    writer.writerow([dept, office, work_name, tender_type, t_id, "", "", "", f"Failed: {e}"])
                    print(f"[WARN] Failed {t_id}: {e}")
                    continue

        print(f"\n✅ Done. Saved: {OUT_CSV.resolve()}")

    finally:
        pass  # keep browser open


if __name__ == "__main__":
    main()


>>> Solve CAPTCHA if it appears, then press ENTER here to continue...  



=== Processing KUWSDB/2025-26/WS/WORK_INDENT336/CALL-3 ===
[OK] KUWSDB/2025-26/WS/WORK_INDENT336/CALL-3 → Success (1 rows).

=== Processing KUWSDB/2024-25/WS/WORK_INDENT310/CALL-2 ===
⚠️ Stale row, retrying...
[OK] KUWSDB/2024-25/WS/WORK_INDENT310/CALL-2 → Success (1 rows).

=== Processing KUWSDB/2024-25/OW/WORK_INDENT325 ===
⚠️ Stale row, retrying...
[OK] KUWSDB/2024-25/OW/WORK_INDENT325 → Success (1 rows).

=== Processing KUWSDB/2024-25/WS/WORK_INDENT256/CALL-4 ===
⚠️ Stale row, retrying...
[OK] KUWSDB/2024-25/WS/WORK_INDENT256/CALL-4 → Success (1 rows).

=== Processing KUWSDB/2024-25/WS/WORK_INDENT288/CALL-2 ===
⚠️ Stale row, retrying...
[OK] KUWSDB/2024-25/WS/WORK_INDENT288/CALL-2 → Success (1 rows).

=== Processing KUWSDB/2024-25/WS/WORK_INDENT324 ===
⚠️ Stale row, retrying...
[OK] KUWSDB/2024-25/WS/WORK_INDENT324 → Success (1 rows).

=== Processing KUWSDB/2024-25/OW/WORK_INDENT312 ===
⚠️ Stale row, retrying...
[OK] KUWSDB/2024-25/OW/WORK_INDENT312 → Success (1 rows).

=== Proces